# Exploration de Payments — phase 1

Analyse du CSV Olist en lecture seule, exclusivement avec le .venv du projet sur C:.

Une ligne représente un enregistrement de paiement, pas une commande ni une échéance individuelle. Clé attendue : (order_id, payment_sequential). payment_installments est distinct du nombre de paiements par commande.

Exécuter les cellules de haut en bas. Les chemins sont relatifs à la racine ou au dossier notebooks. Aucune écriture de données et aucun appel à un validateur.

Le code a été exécuté avec le Python local pendant l'audit du 2026-10-24. Notebook fourni sans sorties enregistrées ni exécution via noyau Jupyter. Résultats et propositions : docs/PAYMENTS_EXPLORATION.md.


In [ ]:
from pathlib import Path
from decimal import Decimal
import hashlib
import json
import platform
import sys

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
payments_path = PROJECT_ROOT / "data" / "olist_order_payments_dataset.csv"
orders_path = PROJECT_ROOT / "data" / "olist_orders_dataset.csv"
source_hashes = {p.name: hashlib.sha256(p.read_bytes()).hexdigest()
                 for p in [payments_path, orders_path]}
payments = pd.read_csv(payments_path)
raw_text = pd.read_csv(payments_path, dtype="string", keep_default_na=False)
orders = pd.read_csv(orders_path, usecols=["order_id", "order_status"])

report = {
    "environment": {"python": platform.python_version(), "pandas": pd.__version__,
                    "executable": sys.executable},
    "source_sha256": source_hashes,
    "source_size_bytes": payments_path.stat().st_size,
    "shape": list(payments.shape),
    "memory_deep_MiB": round(payments.memory_usage(deep=True).sum() / 2**20, 3),
    "dtypes": {name: str(dtype) for name, dtype in payments.dtypes.items()},
    "missing": {name: int(n) for name,n in payments.isna().sum().items()},
    "blank_or_whitespace": {name: int(raw_text[name].str.strip().eq("").sum())
                            for name in payments.columns},
    "exact_duplicate_rows_excess": int(payments.duplicated().sum()),
    "duplicate_key_rows": int(payments.duplicated(
        ["order_id", "payment_sequential"], keep=False).sum()),
    "order_id_format_mismatch": int((~raw_text.order_id.str.fullmatch(
        r"[0-9a-f]{32}", na=False)).sum()),
}
print(json.dumps({key: value for key,value in report.items()
                  if key != "source_sha256"}, ensure_ascii=False, indent=2))


## Nombres et montants
Séparer les valeurs manquantes, non numériques, infinies, négatives et nulles. Les valeurs décimales de payment_value sont normales. Decimal sert à la somme descriptive exacte en BRL ; aucune interprétation comptable.


In [ ]:
numeric_columns = ["payment_sequential", "payment_installments", "payment_value"]
report["numeric_statistics"] = payments[numeric_columns].describe(
    percentiles=[.25, .5, .75, .95, .99]).to_dict()
report["numeric_checks"] = {}
for name in numeric_columns:
    values = pd.to_numeric(payments[name], errors="coerce")
    finite = values.notna() & np.isfinite(values)
    report["numeric_checks"][name] = {
        "missing": int(payments[name].isna().sum()),
        "non_numeric_non_null": int((payments[name].notna() & values.isna()).sum()),
        "non_finite_non_null": int((values.notna() & ~np.isfinite(values)).sum()),
        "negative": int(values.lt(0).sum()),
        "zero": int(values.eq(0).sum()),
        "non_integer_finite": int((finite & values.where(finite).mod(1).ne(0)).sum()),
    }
report["installments_distribution"] = {
    str(value): int(n) for value,n in
    payments.payment_installments.value_counts().sort_index().items()}
report["sequential_distribution"] = {
    str(value): int(n) for value,n in
    payments.payment_sequential.value_counts().sort_index().items()}
report["value_format_mismatch"] = int((~raw_text.payment_value.str.fullmatch(
    r"\d+(?:\.\d{1,2})?", na=False)).sum())
# Decimal is used only for exact descriptive monetary sums, not accounting revenue.
# Malformed source amounts would be reported above and stop this exact sum.
amounts = raw_text.payment_value.map(Decimal)
assert all(value.is_finite() for value in amounts), "Non-finite amount"
report["payment_value_total_BRL"] = str(sum(amounts, Decimal("0")))
print(json.dumps({key: report[key] for key in [
    "numeric_statistics", "numeric_checks", "installments_distribution",
    "sequential_distribution", "payment_value_total_BRL"]}, indent=2))


## Types de paiement
Observer les catégories et leur fréquence. not_defined est une catégorie présente, pas une cellule manquante ; la signaler pour investigation sans la convertir automatiquement en erreur.


In [ ]:
report["payment_types"] = {
    str(value): int(n) for value,n in payments.payment_type.value_counts(
        dropna=False).items()}
type_groups = payments.groupby("payment_type", dropna=False)
report["type_details"] = {}
for name, group in type_groups:
    report["type_details"][str(name)] = {
        "rows": int(len(group)),
        "percent_rows": round(len(group) / len(payments) * 100, 4),
        "zero_values": int(group.payment_value.eq(0).sum()),
        "zero_installments": int(group.payment_installments.eq(0).sum()),
        "installments_gt1": int(group.payment_installments.gt(1).sum()),
        "min_installments": int(group.payment_installments.min()),
        "max_installments": int(group.payment_installments.max()),
        "min_value": float(group.payment_value.min()),
        "max_value": float(group.payment_value.max()),
        "amount_BRL": str(sum(amounts.loc[group.index], Decimal("0"))),
    }
report["type_has_surrounding_whitespace"] = int(
    raw_text.payment_type.ne(raw_text.payment_type.str.strip()).sum())
print(json.dumps(report["type_details"], indent=2))


## Plusieurs paiements et séquences
Les répétitions de order_id sont normales lorsque la clé composite reste unique. Une séquence ne commençant pas à 1 mérite investigation mais ne prouve pas une erreur. Ne pas multiplier payment_value par payment_installments.


In [ ]:
counts = payments.groupby("order_id").size()
key_groups = payments.groupby("order_id").payment_sequential.agg(
    ["size", "min", "max", "nunique"])
sequence_unusual = (
    key_groups["min"].ne(1) | key_groups["max"].ne(key_groups["size"])
    | key_groups["nunique"].ne(key_groups["size"])
)
totals_by_order = amounts.groupby(payments.order_id).sum()
report["multiple_payments"] = {
    "distinct_orders": int(counts.size),
    "orders_with_multiple_rows": int(counts.gt(1).sum()),
    "rows_in_multiple_payment_orders": int(counts[counts.gt(1)].sum()),
    "repeated_order_id_rows_excess": int(payments.order_id.duplicated().sum()),
    "max_rows_per_order": int(counts.max()),
    "row_count_distribution": {str(value): int(n) for value,n in
                               counts.value_counts().sort_index().items()},
    "orders_with_multiple_payment_types": int(
        payments.groupby("order_id").payment_type.nunique().gt(1).sum()),
    "orders_with_unusual_sequence": int(sequence_unusual.sum()),
    "orders_with_sequence_min_not1": int(key_groups["min"].ne(1).sum()),
    "orders_with_sequence_gaps": int((key_groups["nunique"].ne(
        key_groups["max"] - key_groups["min"] + 1)).sum()),
    "orders_with_zero_total": int(totals_by_order.eq(Decimal("0")).sum()),
}
print(json.dumps(report["multiple_payments"], indent=2))


## Références Orders et complétude
La jointure many_to_one vérifie que Orders ne multiplie pas les lignes. L'absence totale de paiement relève du contrôle de complétude côté Orders. Aucun rapprochement direct avec Order Items.


In [ ]:
# A many-to-one join prevents accidental multiplication by duplicate Orders keys.
joined = payments.merge(orders, on="order_id", how="left",
                        validate="many_to_one", indicator=True)
without_payments = orders.loc[~orders.order_id.isin(payments.order_id)]
zero_rows = joined.loc[joined.payment_value.eq(0)]
report["relations"] = {
    "orders_rows": int(len(orders)),
    "orders_duplicate_key_rows": int(orders.order_id.duplicated(keep=False).sum()),
    "joined_rows": int(len(joined)),
    "unknown_order_rows": int(joined["_merge"].eq("left_only").sum()),
    "orders_without_payments": int(len(without_payments)),
    "orders_without_payments_by_status": {
        str(s): int(n) for s,n in without_payments.order_status.value_counts().items()},
    "zero_payment_rows_by_status": {
        str(s): int(n) for s,n in zero_rows.order_status.value_counts().items()},
    "zero_payment_distinct_orders": int(zero_rows.order_id.nunique()),
}
assert source_hashes == {p.name: hashlib.sha256(p.read_bytes()).hexdigest()
                         for p in [payments_path, orders_path]}, "Source changed"
print(json.dumps(report["relations"], indent=2))
# No payment timestamp exists in these columns: do not invent event-time KPIs.
# Never multiply payment_value by payment_installments.
# For future Items reconciliation, aggregate both tables by order_id separately.



## Avant de créer le validateur
Proposer un contrat de schéma, clé composite, références, valeurs numériques finies et compteurs entiers ; garder à part les montants nuls, not_defined, zéro échéance, séquences inhabituelles, extrêmes et commandes sans paiements.
Aucun payments_validator.py créé. Aucun statut valid/review attribué par cette exploration. Pas de date de paiement dans le CSV ; pas de revenu comptable, remboursement ou UTC inventé. Attendre validation des règles.
